# Linux Compression and Archiving

## Expanded Educational Notebook

This notebook explains:

- compression versus archiving,
- lossless versus lossy compression,
- why some files compress well and others do not,
- `gzip`,
- `bzip2`,
- `xz`,
- common options,
- compression levels,
- integrity testing,
- standard input and standard output,
- pipes,
- recursive compression,
- `tar` integration,
- practical use cases,
- security and reliability notes,
- hands-on labs,
- review questions and answers.

The examples are designed for Linux systems.

## Learning objectives

After completing this notebook, you should be able to:

1. explain the difference between compression and archiving,
2. distinguish lossless and lossy compression,
3. identify file formats that are already compressed,
4. use `gzip`, `gunzip`, and `zcat`,
5. use `bzip2`, `bunzip2`, and `bzcat`,
6. use `xz`, `unxz`, and `xzcat`,
7. compare speed and compression ratio,
8. use compression levels,
9. compress without deleting the original,
10. write compressed output to standard output,
11. test compressed-file integrity,
12. combine `tar` with gzip, bzip2, and xz,
13. choose an appropriate compression tool.

# Part I — Core Concepts

## 1. What is compression?

Compression reduces the amount of storage needed to represent data.

Suppose a text file contains:

```text
AAAAAAAAAAAAAAAAAAAA
```

Instead of storing every `A` separately, a compression algorithm may represent the repeated pattern more efficiently.

The compressed representation is usually not human-readable.

Compression can reduce:

- disk usage,
- network transfer time,
- backup size,
- package-download size,
- log storage.

## 2. What is archiving?

Archiving combines multiple files and directories into one container while preserving structure and often metadata.

For example:

```text
project/
├── README.md
├── src/
│   └── main.c
└── data/
    └── input.csv
```

A `tar` archive can store this entire tree in one file:

```text
project.tar
```

An archive may preserve:

- filenames,
- directory structure,
- permissions,
- ownership,
- timestamps,
- symbolic links,
- hard-link relationships.

## 3. Compression versus archiving

These are different operations.

### Compression

```text
large file -> smaller compressed file
```

Examples:

- `gzip`
- `bzip2`
- `xz`

### Archiving

```text
many files/directories -> one archive file
```

Example:

- `tar`

### Archive plus compression

```text
many files -> tar archive -> compressed archive
```

Examples:

```text
project.tar.gz
project.tar.bz2
project.tar.xz
```

## 4. Why does Linux often use `tar` first?

Traditional compression tools usually operate on one byte stream.

A directory is not simply one sequence of file contents. It also contains:

- names,
- subdirectories,
- metadata,
- links,
- ordering.

`tar` converts a directory tree into one stream.  
Then a compressor can compress that stream.

Conceptually:

```text
folder
  |
  v
tar archive
  |
  v
gzip / bzip2 / xz
```

## 5. Example: archive only

```bash
tar -cf project.tar project/
```

Meaning:

- `c` = create,
- `f` = archive filename follows,
- no compression option is used.

The result is one archive file, but it may not be much smaller than the original data.

## 6. Example: archive and compress

```bash
tar -czf project.tar.gz project/
```

Meaning:

- `c` = create archive,
- `z` = compress using gzip,
- `f` = output filename.

Similarly:

```bash
tar -cjf project.tar.bz2 project/
tar -cJf project.tar.xz project/
```

# Part II — Lossless and Lossy Compression

## 7. What is lossless compression?

Lossless compression allows the original data to be reconstructed exactly.

After decompression:

```text
original bytes == restored bytes
```

This is required for:

- source code,
- executable files,
- configuration files,
- databases,
- documents,
- scientific data,
- backups.

## 8. Lossless examples

Common lossless formats and tools include:

- gzip,
- bzip2,
- xz,
- ZIP,
- 7z,
- PNG,
- FLAC,
- lossless WebP,
- Zstandard.

If a single bit changes after decompression, the data may be unusable, so exact restoration matters.

## 9. What is lossy compression?

Lossy compression permanently discards some information to achieve a smaller size.

The restored data is similar to the original, but not byte-for-byte identical.

Lossy compression is useful when human perception allows approximation.

## 10. Lossy examples

Examples include:

- JPEG images,
- MP3 audio,
- AAC audio,
- Opus audio,
- H.264 video,
- H.265/HEVC video,
- AV1 video,
- lossy WebP.

A lossy encoder may remove details that are considered less noticeable.

## 11. Lossless versus lossy

| Property | Lossless | Lossy |
|---|---|---|
| Exact reconstruction | Yes | No |
| Suitable for source code | Yes | No |
| Suitable for executables | Yes | No |
| Typical size reduction | Moderate to high | Often very high |
| Common data | Text, code, backups | Audio, video, photos |
| Examples | gzip, xz, PNG, FLAC | JPEG, MP3, H.264 |

## 12. Why JPEG and MP3 usually do not compress much with gzip

JPEG and MP3 are already compressed.

Their remaining byte patterns contain less easy redundancy.

Therefore:

```bash
gzip photo.jpg
```

may produce:

- almost no size reduction,
- or even a slightly larger file because the gzip format adds headers and checksums.

The same is often true for:

- `.mp3`,
- `.mp4`,
- `.zip`,
- `.gz`,
- `.xz`,
- `.7z`,
- many PDFs,
- PNG images.

## 13. Files that often compress well

These often contain repeated structures:

- plain text,
- source code,
- CSV,
- JSON,
- XML,
- logs,
- SQL dumps,
- repetitive scientific data,
- uncompressed bitmap images.

## 14. Files that often compress poorly

These are commonly already compressed or high-entropy:

- JPEG,
- MP3,
- MP4,
- ZIP,
- gzip,
- xz,
- encrypted files,
- random data.

Encrypted data is intentionally designed to look random, so compression normally provides little benefit.

# Part III — Measuring Compression

## 15. Compression ratio

One common definition is:

```text
compression ratio = compressed size / original size
```

Smaller is better.

Example:

```text
original = 100 MB
compressed = 25 MB
ratio = 0.25
```

This means the compressed file is 25% of the original size.

## 16. Space saving percentage

Another useful calculation:

```text
saving = 1 - (compressed size / original size)
```

Example:

```text
original = 100 MB
compressed = 25 MB
saving = 75%
```

## 17. Compression trade-offs

Compression tools differ in:

- compression speed,
- decompression speed,
- resulting size,
- memory usage,
- compatibility,
- CPU cost.

A smaller output is not always the best result.

For frequently accessed logs, fast compression and decompression may matter more than maximum compression.

# Part IV — gzip

## 18. What is gzip?

`gzip` is a widely used lossless compression utility.

Its name means:

> GNU zip

It uses the DEFLATE algorithm.

Typical characteristics:

- fast,
- widely available,
- fast decompression,
- moderate compression ratio,
- `.gz` extension,
- designed primarily for one file or one stream.

## 19. Basic gzip command

```bash
gzip file.txt
```

Usually this:

1. creates `file.txt.gz`,
2. removes `file.txt`.

This replacement behavior surprises many beginners.

## 20. Decompressing gzip files

```bash
gzip -d file.txt.gz
```

Equivalent command:

```bash
gunzip file.txt.gz
```

Usually this restores `file.txt` and removes `file.txt.gz`.

## 21. `gzip -k`

Keep the original file:

```bash
gzip -k file.txt
```

Afterward:

```text
file.txt
file.txt.gz
```

This is useful in educational labs and whenever the original should remain.

## 22. `gzip -c`

`-c` writes compressed data to standard output instead of replacing the source.

```bash
gzip -c file.txt > file.txt.gz
```

Benefits:

- original remains,
- output filename is controlled by shell redirection,
- useful in pipelines,
- useful for streaming.

## 23. Decompress to standard output

```bash
gzip -dc file.txt.gz
```

or:

```bash
zcat file.txt.gz
```

This displays decompressed contents without creating a restored file.

## 24. Important gzip options

| Option | Meaning |
|---|---|
| `-d` | decompress |
| `-c` | write to standard output |
| `-k` | keep original input |
| `-f` | force overwrite or special handling |
| `-v` | verbose output |
| `-q` | quiet |
| `-r` | recursively process directories |
| `-l` | list compressed-file information |
| `-t` | test integrity |
| `-1` | fastest compression |
| `-9` | strongest compression |
| `-n` | do not store original name/time |
| `-N` | restore/use stored name and time when possible |

## 25. Compression levels in gzip

gzip supports levels:

```text
-1 through -9
```

General interpretation:

- `-1` = fastest, larger output,
- `-6` = typical default,
- `-9` = slower, smaller output.

Example:

```bash
gzip -1 -k file.txt
gzip -9 -k file.txt
```

The size difference may be small or large depending on the data.

## 26. `gzip -l`

List information:

```bash
gzip -l file.txt.gz
```

Typical fields:

- compressed size,
- uncompressed size,
- compression ratio,
- original name.

## 27. `gzip -t`

Test compressed-file integrity:

```bash
gzip -t file.txt.gz
```

No output usually means success.

Verbose test:

```bash
gzip -tv file.txt.gz
```

This is useful before deleting the original or transferring backups.

## 28. Recursive gzip

```bash
gzip -r directory/
```

This compresses individual eligible files recursively.

It does **not** create one archive preserving the entire tree as one object.

You may get:

```text
directory/a.txt.gz
directory/sub/b.txt.gz
```

For backups or distribution, `tar` is usually more appropriate.

## 29. `zcat`, `zless`, and `zgrep`

Useful tools:

```bash
zcat file.gz
zless file.gz
zgrep ERROR application.log.gz
```

They allow reading and searching gzip-compressed data without manually decompressing it first.

# Part V — bzip2

## 30. What is bzip2?

`bzip2` is a lossless compression utility.

Typical characteristics:

- often compresses better than gzip,
- usually slower than gzip,
- uses more CPU,
- output extension `.bz2`,
- based mainly on the Burrows-Wheeler transform and related coding techniques.

## 31. Basic bzip2 command

```bash
bzip2 file.txt
```

Usually this creates:

```text
file.txt.bz2
```

and removes the original.

## 32. Decompressing bzip2 files

```bash
bzip2 -d file.txt.bz2
```

Equivalent:

```bash
bunzip2 file.txt.bz2
```

## 33. Keep original with bzip2

```bash
bzip2 -k file.txt
```

This retains the source file.

## 34. Write bzip2 data to standard output

```bash
bzip2 -c file.txt > file.txt.bz2
```

Decompress to standard output:

```bash
bzip2 -dc file.txt.bz2
```

or:

```bash
bzcat file.txt.bz2
```

## 35. Important bzip2 options

| Option | Meaning |
|---|---|
| `-d` | decompress |
| `-c` | standard output |
| `-k` | keep original |
| `-f` | force overwrite |
| `-t` | test integrity |
| `-v` | verbose |
| `-1` to `-9` | block-size/compression setting |
| `-q` | quiet |

## 36. bzip2 levels

For bzip2, `-1` through `-9` mainly control block size.

Higher settings often:

- use more memory,
- may improve compression,
- may take longer.

The meaning is not identical to gzip's internal strategy levels.

## 37. Test bzip2 integrity

```bash
bzip2 -t file.txt.bz2
```

Verbose:

```bash
bzip2 -tv file.txt.bz2
```

## 38. bzip2 companion commands

```bash
bzcat file.bz2
bzless file.bz2
bzgrep pattern file.bz2
```

Availability may vary by distribution.

# Part VI — xz

## 39. What is xz?

`xz` is a lossless compression utility based on LZMA2.

Typical characteristics:

- excellent compression ratio,
- slower compression,
- relatively fast decompression,
- greater memory use,
- `.xz` extension,
- common for source distributions and package repositories.

## 40. Basic xz command

```bash
xz file.txt
```

Usually creates:

```text
file.txt.xz
```

and removes the original.

## 41. Decompressing xz files

```bash
xz -d file.txt.xz
```

Equivalent:

```bash
unxz file.txt.xz
```

## 42. Keep original with xz

```bash
xz -k file.txt
```

## 43. Standard output with xz

Compress:

```bash
xz -c file.txt > file.txt.xz
```

Decompress:

```bash
xz -dc file.txt.xz
```

or:

```bash
xzcat file.txt.xz
```

## 44. Important xz options

| Option | Meaning |
|---|---|
| `-d` | decompress |
| `-c` | standard output |
| `-k` | keep original |
| `-f` | force |
| `-t` | test integrity |
| `-l` | list information |
| `-v` | verbose |
| `-1` to `-9` | compression presets |
| `-0` | very fast preset |
| `-e` | extreme mode |
| `-T` | thread count |

## 45. xz compression levels

Common presets:

```text
-0 through -9
```

General behavior:

- low levels = faster, less memory, larger output,
- high levels = slower, more memory, smaller output.

Example:

```bash
xz -1 -k file.txt
xz -9 -k file.txt
```

## 46. Extreme mode

```bash
xz -9e file.txt
```

`-e` asks xz to use a more exhaustive strategy.

It may:

- increase compression time significantly,
- use more CPU,
- produce only a modest improvement.

Use it when maximum size reduction matters more than time.

## 47. Multithreaded xz

```bash
xz -T0 file.txt
```

`-T0` usually means use as many threads as the tool considers appropriate.

Important:

- multithreading may increase memory usage,
- output characteristics can vary,
- older xz versions had limitations compared with newer ones.

## 48. Test xz integrity

```bash
xz -t file.txt.xz
```

Verbose:

```bash
xz -tv file.txt.xz
```

## 49. List xz information

```bash
xz -l file.txt.xz
```

This may show:

- compressed size,
- uncompressed size,
- ratio,
- number of streams,
- check type.

## 50. xz companion commands

```bash
xzcat file.xz
xzless file.xz
xzgrep pattern file.xz
```

# Part VII — Comparing gzip, bzip2, and xz

## 51. General comparison

| Tool | Compression speed | Decompression speed | Compression ratio | Extension |
|---|---|---|---|---|
| gzip | Fast | Very fast | Good | `.gz` |
| bzip2 | Slower | Moderate | Better than gzip in many cases | `.bz2` |
| xz | Slowest | Often reasonably fast | Usually best | `.xz` |

## 52. Typical use cases

### gzip

Good for:

- logs,
- web transfer,
- broad compatibility,
- fast compression/decompression.

### bzip2

Good for:

- older workflows,
- text-heavy data where better compression than gzip is useful,
- systems where bzip2 is already standard.

### xz

Good for:

- software source releases,
- long-term storage,
- package repositories,
- situations where smaller files matter more than compression time.

## 53. Which one is “best”?

There is no universal best tool.

Choose according to:

- file type,
- CPU availability,
- memory limits,
- transfer cost,
- storage cost,
- compatibility,
- how often decompression occurs,
- how quickly output is needed.

# Part VIII — tar Integration

## 54. Create a plain tar archive

```bash
tar -cf archive.tar folder/
```

## 55. List tar contents

```bash
tar -tf archive.tar
```

Verbose listing:

```bash
tar -tvf archive.tar
```

## 56. Extract a tar archive

```bash
tar -xf archive.tar
```

## 57. tar with gzip

Create:

```bash
tar -czf archive.tar.gz folder/
```

List:

```bash
tar -tzf archive.tar.gz
```

Extract:

```bash
tar -xzf archive.tar.gz
```

## 58. tar with bzip2

Create:

```bash
tar -cjf archive.tar.bz2 folder/
```

List:

```bash
tar -tjf archive.tar.bz2
```

Extract:

```bash
tar -xjf archive.tar.bz2
```

## 59. tar with xz

Create:

```bash
tar -cJf archive.tar.xz folder/
```

List:

```bash
tar -tJf archive.tar.xz
```

Extract:

```bash
tar -xJf archive.tar.xz
```

## 60. Extract to a chosen directory

```bash
mkdir restore
tar -xzf archive.tar.gz -C restore/
```

`-C` changes directory before extraction.

## 61. Extract one file

```bash
tar -xzf archive.tar.gz folder/path/file.txt
```

The path must normally match the archive entry exactly.

## 62. Exclude files

```bash
tar -czf backup.tar.gz \
    --exclude='*.tmp' \
    --exclude='cache/' \
    project/
```

Be careful with shell expansion and archive-relative paths.

## 63. Preserve permissions and ownership

`tar` stores metadata, but restoration depends on:

- current user,
- root privileges,
- filesystem support,
- tar options,
- security restrictions.

An ordinary user cannot normally restore arbitrary ownership.

## 64. Why `.tar.gz` has two extensions

```text
project.tar.gz
```

means:

1. `project.tar` is the archive,
2. gzip compressed the tar stream.

Likewise:

```text
.tar.bz2
.tar.xz
```

## 65. `.tgz`

`.tgz` is a shorter name for `.tar.gz`.

Example:

```text
backup.tgz
```

It generally means the same archive format combination.

# Part IX — Streams and Pipes

## 66. Standard input and standard output

Compression tools can read from standard input and write to standard output.

Example:

```bash
cat file.txt | gzip > file.txt.gz
```

More directly:

```bash
gzip -c file.txt > file.txt.gz
```

## 67. Stream a tar archive through gzip

```bash
tar -cf - folder/ | gzip > archive.tar.gz
```

Here:

- `tar -cf -` writes archive data to standard output,
- `gzip` compresses that stream,
- `>` saves the result.

## 68. Decompress a stream into tar

```bash
gzip -dc archive.tar.gz | tar -xf -
```

This is conceptually equivalent to:

```bash
tar -xzf archive.tar.gz
```

## 69. Compress over SSH

Conceptual example:

```bash
tar -cf - project/ | gzip | ssh user@server 'cat > project.tar.gz'
```

Or restore remotely:

```bash
ssh user@server 'gzip -dc project.tar.gz' | tar -xf -
```

Use carefully with:

- authentication,
- destination paths,
- permissions,
- network reliability.

# Part X — Practical Reliability

## 70. Verify before deleting originals

A safer workflow:

```bash
gzip -c important.txt > important.txt.gz
gzip -t important.txt.gz
```

Only after verification should the original be removed if appropriate.

## 71. Checksums

Compression formats include integrity checks, but an external checksum is useful for transfer verification.

Example:

```bash
sha256sum backup.tar.xz > backup.tar.xz.sha256
sha256sum -c backup.tar.xz.sha256
```

## 72. Corruption behavior

Compressed files may become partially or completely unreadable if corrupted.

Consequences depend on:

- format,
- location of damage,
- block/stream structure,
- recovery tools,
- redundancy.

Compression is not a backup strategy by itself.

## 73. Compression is not encryption

Compression reduces size.

Encryption protects confidentiality.

A `.gz`, `.bz2`, or `.xz` file is not secret merely because it is unreadable in a text editor.

## 74. Compression bombs

A compression bomb is a small compressed file that expands into a huge amount of data.

Risks include:

- filling disks,
- exhausting memory,
- consuming CPU,
- denial of service.

Inspect untrusted archives carefully and extract in restricted environments when appropriate.

# Part XI — Hands-On Labs

## 75. Lab setup

The following commands create a temporary laboratory directory.

In [ ]:
rm -rf compression_expanded_lab
mkdir compression_expanded_lab
cd compression_expanded_lab

yes "Linux compression is useful for repetitive text." | head -n 5000 > repetitive.txt

python3 - <<'PY'
import os
with open("random.bin", "wb") as f:
    f.write(os.urandom(200000))
PY

ls -lh

## 76. Lab: compare compressibility

`repetitive.txt` should compress well.

`random.bin` should compress poorly because random bytes contain little exploitable redundancy.

In [ ]:
gzip -k repetitive.txt
gzip -k random.bin

ls -lh repetitive.txt repetitive.txt.gz random.bin random.bin.gz

## 77. Lab: compare gzip levels

In [ ]:
gzip -1 -c repetitive.txt > repetitive-gzip-1.gz
gzip -9 -c repetitive.txt > repetitive-gzip-9.gz

ls -lh repetitive.txt repetitive-gzip-1.gz repetitive-gzip-9.gz

## 78. Lab: test gzip integrity

In [ ]:
gzip -tv repetitive-gzip-9.gz

## 79. Lab: inspect gzip information

In [ ]:
gzip -l repetitive-gzip-9.gz

## 80. Lab: bzip2

In [ ]:
bzip2 -k repetitive.txt
bzip2 -t repetitive.txt.bz2

ls -lh repetitive.txt repetitive.txt.bz2

## 81. Lab: xz

In [ ]:
xz -k repetitive.txt
xz -t repetitive.txt.xz

ls -lh repetitive.txt repetitive.txt.xz
xz -l repetitive.txt.xz

## 82. Lab: compare all three

In [ ]:
ls -lh repetitive.txt repetitive.txt.gz repetitive.txt.bz2 repetitive.txt.xz

## 83. Lab: display compressed content

In [ ]:
echo "--- zcat ---"
zcat repetitive.txt.gz | head -n 3

echo "--- bzcat ---"
bzcat repetitive.txt.bz2 | head -n 3

echo "--- xzcat ---"
xzcat repetitive.txt.xz | head -n 3

## 84. Lab: create a directory tree

In [ ]:
mkdir -p project/src project/data

printf '#include <stdio.h>\nint main(){return 0;}\n' > project/src/main.c
printf 'name,value\nalpha,1\nbeta,2\n' > project/data/sample.csv
printf 'Compression project\n' > project/README.md

find project -type f -print

## 85. Lab: create plain tar archive

In [ ]:
tar -cf project.tar project
tar -tf project.tar
ls -lh project.tar

## 86. Lab: create tar.gz

In [ ]:
tar -czf project.tar.gz project
tar -tzf project.tar.gz
ls -lh project.tar.gz

## 87. Lab: create tar.bz2

In [ ]:
tar -cjf project.tar.bz2 project
tar -tjf project.tar.bz2
ls -lh project.tar.bz2

## 88. Lab: create tar.xz

In [ ]:
tar -cJf project.tar.xz project
tar -tJf project.tar.xz
ls -lh project.tar.xz

## 89. Lab: compare archive sizes

In [ ]:
ls -lh project.tar project.tar.gz project.tar.bz2 project.tar.xz

## 90. Lab: extract into separate directories

In [ ]:
mkdir restore-gz restore-bz2 restore-xz

tar -xzf project.tar.gz -C restore-gz
tar -xjf project.tar.bz2 -C restore-bz2
tar -xJf project.tar.xz -C restore-xz

find restore-gz restore-bz2 restore-xz -type f -print

## 91. Lab: checksum original and restored files

In [ ]:
sha256sum project/README.md restore-gz/project/README.md
sha256sum project/src/main.c restore-bz2/project/src/main.c
sha256sum project/data/sample.csv restore-xz/project/data/sample.csv

# Part XII — Common Mistakes

## 92. Mistake: thinking gzip archives directories

`gzip` compresses files or streams.

It does not create a structured archive of a directory tree.

Use:

```bash
tar -czf archive.tar.gz directory/
```

## 93. Mistake: forgetting that compression may remove the original

Commands such as:

```bash
gzip file
bzip2 file
xz file
```

normally replace the source.

Use `-k` or `-c` when you need to keep it.

## 94. Mistake: recompressing compressed media

Running gzip on JPEG or MP3 usually wastes time and gains little.

Always compare sizes rather than assuming compression helps.

## 95. Mistake: using maximum compression everywhere

`-9` may provide only a small improvement but cost much more CPU time.

Use realistic benchmarks for your workload.

## 96. Mistake: confusing archive extension and actual format

A file named:

```text
backup.tar.gz
```

is probably gzip-compressed tar, but filenames can lie.

Use:

```bash
file backup.tar.gz
```

and test commands.

## 97. Mistake: extracting untrusted archives carelessly

Archives may contain:

- absolute paths,
- `..` path traversal,
- symbolic links,
- unusual permissions,
- huge expanded data.

Inspect first:

```bash
tar -tf archive.tar.gz
```

Extract into a controlled directory.

# Part XIII — Review Questions

## 98. Questions

1. What is compression?
2. What is archiving?
3. Why are compression and archiving different?
4. Why is `tar` commonly used before gzip?
5. What is lossless compression?
6. What is lossy compression?
7. Give three lossless examples.
8. Give three lossy examples.
9. Why does JPEG usually not compress much with gzip?
10. Why does encrypted data compress poorly?
11. What does `gzip file.txt` normally do?
12. How do you keep the original with gzip?
13. What does `gzip -c` do?
14. What does `gzip -d` do?
15. What is `gunzip`?
16. What is `zcat`?
17. What does `gzip -t` do?
18. What does `gzip -l` show?
19. What is the difference between gzip `-1` and `-9`?
20. What is bzip2?
21. What extension does bzip2 use?
22. How do you decompress `.bz2`?
23. What is `bzcat`?
24. What is xz?
25. What extension does xz use?
26. How do you keep the original with xz?
27. What does `xz -T0` mean?
28. What is xz extreme mode?
29. Which generally compresses fastest?
30. Which generally gives the smallest output?
31. How do you create a `.tar.gz` archive?
32. How do you extract a `.tar.xz` archive?
33. What does `tar -tf` do?
34. What does `tar -C` do?
35. Why is `.tar.gz` a double extension?
36. Is compression encryption?
37. What is a compression bomb?
38. Why should compressed backups be tested?
39. Why are checksums useful?
40. Why is `gzip -r` different from `tar -czf`?

# Part XIV — Answers

## 99. Answers

1. It reduces the number of bytes used to represent data.
2. It combines multiple files/directories and metadata into one container.
3. One reduces size; the other organizes multiple objects into one archive.
4. `tar` converts a directory tree into one byte stream.
5. Compression that reconstructs the exact original bytes.
6. Compression that permanently discards some information.
7. gzip, bzip2, xz.
8. JPEG, MP3, H.264.
9. JPEG is already compressed and has little remaining redundancy.
10. Encryption makes output resemble random data.
11. It creates `file.txt.gz` and usually removes `file.txt`.
12. Use `gzip -k file.txt` or `gzip -c file.txt > file.txt.gz`.
13. It writes compressed output to standard output.
14. It decompresses.
15. A common command equivalent to gzip decompression.
16. It writes decompressed gzip content to standard output.
17. It tests gzip-file integrity.
18. Compressed size, uncompressed size, ratio, and related information.
19. `-1` favors speed; `-9` favors compression ratio.
20. A lossless compression utility often producing smaller files than gzip but more slowly.
21. `.bz2`.
22. `bzip2 -d file.bz2` or `bunzip2 file.bz2`.
23. It writes decompressed bzip2 content to standard output.
24. A lossless LZMA2-based compression utility.
25. `.xz`.
26. `xz -k file`.
27. Use automatic/all available threading according to xz behavior.
28. A more exhaustive, slower compression strategy enabled with `-e`.
29. gzip, in general.
30. xz, in general.
31. `tar -czf archive.tar.gz folder/`.
32. `tar -xJf archive.tar.xz`.
33. Lists archive contents.
34. Changes directory for an operation such as extraction.
35. It is a tar archive compressed by gzip.
36. No.
37. A small compressed input that expands to an enormous size.
38. To detect corruption before originals are deleted or backups are needed.
39. They verify that transferred or stored bytes match.
40. `gzip -r` compresses files individually; `tar -czf` creates one compressed archive preserving the tree.

# Part XV — Cheat Sheet

## 100. gzip cheat sheet

```bash
gzip file
gzip -k file
gzip -c file > file.gz
gzip -d file.gz
gunzip file.gz
zcat file.gz
gzip -l file.gz
gzip -t file.gz
gzip -1 file
gzip -9 file
```

## 101. bzip2 cheat sheet

```bash
bzip2 file
bzip2 -k file
bzip2 -c file > file.bz2
bzip2 -d file.bz2
bunzip2 file.bz2
bzcat file.bz2
bzip2 -t file.bz2
```

## 102. xz cheat sheet

```bash
xz file
xz -k file
xz -c file > file.xz
xz -d file.xz
unxz file.xz
xzcat file.xz
xz -l file.xz
xz -t file.xz
xz -9e file
xz -T0 file
```

## 103. tar cheat sheet

```bash
tar -cf archive.tar folder/
tar -xf archive.tar
tar -tf archive.tar

tar -czf archive.tar.gz folder/
tar -xzf archive.tar.gz
tar -tzf archive.tar.gz

tar -cjf archive.tar.bz2 folder/
tar -xjf archive.tar.bz2

tar -cJf archive.tar.xz folder/
tar -xJf archive.tar.xz
```

## 104. Final comparison

| Goal | Recommended starting point |
|---|---|
| Fast general compression | gzip |
| Better compression, older established tool | bzip2 |
| Maximum compression among these three | xz |
| Compress a directory tree | tar + compressor |
| Read compressed text | zcat / bzcat / xzcat |
| Verify integrity | `-t` |
| Keep original | `-k` |
| Stream output | `-c` |